# Preparacao e Pre-processamento de Dados para LTR

Este notebook gera dois datasets finais com as colunas obrigatorias:
`['query_id', 'doc_id', 'relevance_label', 'query_text', 'passage_text']`.

Saidas:
- `data/msmarco_v2_train_candidates_with_text.csv`
- `data/trec_dl_2023_candidates_with_text.csv`

Restricao atendida: sem download local do corpus bruto completo; o texto das passagens e recuperado por DuckDB a partir de Parquet remoto.

In [1]:
from __future__ import annotations

import os
import random
from pathlib import Path

import duckdb
import ir_datasets
import pandas as pd
from tqdm import tqdm

In [2]:
# Configuracoes principais
RANDOM_SEED = 42
TRAIN_QUERY_SAMPLE_SIZE = 1000

# Caminho de saida
OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_OUT = OUTPUT_DIR / 'msmarco_v2_train_candidates_with_text.csv'
TREC_OUT = OUTPUT_DIR / 'trec_dl_2023_candidates_with_text.csv'

# Token HF para acesso a datasets gated (se aplicavel).
HF_TOKEN = os.getenv('HF_TOKEN', '').strip() or os.getenv('HUGGING_FACE_HUB_TOKEN', '').strip()

# Fallback interativo: pede token somente em memoria (nao salva no notebook).
if not HF_TOKEN:
    try:
        from getpass import getpass
        _tmp = getpass('HF_TOKEN nao encontrado. Cole um token Read do Hugging Face (ou Enter para pular): ')
        HF_TOKEN = (_tmp or '').strip()
    except Exception:
        pass

# Parquet remoto do corpus de passagens
# Prioridade: use MSMARCO_V2_PASSAGES_PARQUET para apontar para seu mirror remoto.
# Exemplo: export MSMARCO_V2_PASSAGES_PARQUET='s3://.../*.parquet' ou 'https://.../*.parquet'
REMOTE_PARQUET_CANDIDATES = [
    os.getenv('MSMARCO_V2_PASSAGES_PARQUET', '').strip(),
    'hf://datasets/irds/msmarco-passage-v2/docs/*.parquet',
    'hf://datasets/irds/msmarco-passage-v2/**/*.parquet',
]
REMOTE_PARQUET_CANDIDATES = [p for p in REMOTE_PARQUET_CANDIDATES if p]

print('Sample de queries treino:', TRAIN_QUERY_SAMPLE_SIZE)
print('HF_TOKEN configurado:', bool(HF_TOKEN))
print('Candidatos de Parquet remoto (ordem de tentativa):')
for p in REMOTE_PARQUET_CANDIDATES:
    print(' -', p)

Sample de queries treino: 1000
HF_TOKEN configurado: True
Candidatos de Parquet remoto (ordem de tentativa):
 - hf://datasets/irds/msmarco-passage-v2/docs/*.parquet
 - hf://datasets/irds/msmarco-passage-v2/**/*.parquet


In [3]:
def reservoir_sample_queries(dataset, k: int, seed: int = 42) -> pd.DataFrame:
    """Amostra k queries unicas com reservoir sampling para evitar carregar tudo em memoria."""
    rng = random.Random(seed)
    sample = []

    for i, q in enumerate(dataset.queries_iter(), start=1):
        row = {'query_id': str(q.query_id), 'query_text': q.text}
        if i <= k:
            sample.append(row)
        else:
            j = rng.randint(1, i)
            if j <= k:
                sample[j - 1] = row

    df = pd.DataFrame(sample).drop_duplicates(subset=['query_id'], keep='first')
    if len(df) < k:
        raise RuntimeError(f'Nao foi possivel obter {k} queries unicas, obtido={len(df)}')
    return df.head(k).reset_index(drop=True)

dataset_train = ir_datasets.load('msmarco-passage-v2/train')
df_train_queries = reservoir_sample_queries(dataset_train, TRAIN_QUERY_SAMPLE_SIZE, RANDOM_SEED)
train_qid_set = set(df_train_queries['query_id'])

print('Queries de treino amostradas:', len(df_train_queries))
df_train_queries.head(3)

Queries de treino amostradas: 1000


,query_id,query_text
0,304409,how much did usa buy alaska for
1,222999,how do you make a photo quilt
2,919810,what was the claude glass


In [4]:
# Candidatos de treino: qrels das 1000 queries amostradas
train_rows = []
for r in tqdm(dataset_train.qrels_iter(), desc='Filtrando qrels train'):
    qid = str(r.query_id)
    if qid in train_qid_set:
        train_rows.append({
            'query_id': qid,
            'doc_id': str(r.doc_id),
            'relevance_label': int(r.relevance),
        })

df_train_candidates = pd.DataFrame(train_rows)
df_train_candidates = df_train_candidates.merge(df_train_queries, on='query_id', how='left')

print('Registros treino (qrels filtrados):', len(df_train_candidates))
print('Queries treino cobertas:', df_train_candidates['query_id'].nunique())
df_train_candidates.head(3)

Filtrando qrels train: 284212it [00:00, 972100.69it/s]

Registros treino (qrels filtrados): 1026
Queries treino cobertas: 1000


,query_id,doc_id,relevance_label,query_text
0,693161,msmarco_passage_09_122268194,1,what is a nrr noise reduction rating mean
1,396846,msmarco_passage_19_374592452,1,integrated dna technology
2,629870,msmarco_passage_04_254329123,1,what does a physical therapist do?


In [5]:
# Avaliacao: TREC DL 2023 oficial (queries + qrels)
dataset_dl23 = ir_datasets.load('msmarco-passage-v2/trec-dl-2023/judged')

dl23_queries = []
for q in dataset_dl23.queries_iter():
    dl23_queries.append({'query_id': str(q.query_id), 'query_text': q.text})
df_dl23_queries = pd.DataFrame(dl23_queries)

dl23_rows = []
for r in dataset_dl23.qrels_iter():
    dl23_rows.append({
        'query_id': str(r.query_id),
        'doc_id': str(r.doc_id),
        'relevance_label': int(r.relevance),
    })

df_dl23_candidates = pd.DataFrame(dl23_rows).merge(df_dl23_queries, on='query_id', how='left')

print('Queries DL23:', len(df_dl23_queries))
print('Registros DL23 (qrels):', len(df_dl23_candidates))
df_dl23_candidates.head(3)

Queries DL23: 82
Registros DL23 (qrels): 22327


,query_id,doc_id,relevance_label,query_text
0,2001010,msmarco_passage_00_257661787,0,cost comparison of funerals in australia
1,2001010,msmarco_passage_00_257662430,0,cost comparison of funerals in australia
2,2001010,msmarco_passage_00_257664103,0,cost comparison of funerals in australia


In [6]:
# Recuperacao de textos de passagens via DuckDB + Parquet remoto (streaming)
needed_doc_ids = sorted(set(df_train_candidates['doc_id']) | set(df_dl23_candidates['doc_id']))
needed_ids_df = pd.DataFrame({'doc_id': needed_doc_ids})
print('Total de doc_ids unicos para buscar texto:', len(needed_ids_df))

# Fail-fast para evitar erro pouco informativo com fontes hf:// gated
if any(p.startswith('hf://') for p in REMOTE_PARQUET_CANDIDATES) and not HF_TOKEN:
    print('Aviso: fontes hf:// detectadas sem HF_TOKEN.')
    print('Se o dataset for gated, configure: export HF_TOKEN=seu_token_hf')

con = duckdb.connect(database=':memory:')
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')

# Algumas versoes usam extensao hf dedicada; se nao existir, seguimos com httpfs.
try:
    con.execute('INSTALL hf;')
    con.execute('LOAD hf;')
except Exception as _hf_ext_err:
    print('Extensao hf indisponivel, seguindo com httpfs.')

# Propaga token para o processo e para configuracoes DuckDB.
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
    safe_token = HF_TOKEN.replace("'", "''")
    try:
        con.execute(f"SET hf_token='{safe_token}'")
    except Exception as _set_err:
        print('SET hf_token nao suportado nesta versao de DuckDB.')
    try:
        con.execute(
            f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{safe_token}')"
        )
        print('Secret HUGGINGFACE configurado no DuckDB.')
    except Exception as _secret_err:
        print('CREATE SECRET HUGGINGFACE nao suportado nesta versao de DuckDB.')

con.register('needed_ids', needed_ids_df)

def detect_columns(conn, parquet_glob: str) -> tuple[str, str]:
    cols = conn.execute(
        f"SELECT * FROM read_parquet('{parquet_glob}') LIMIT 0"
    ).df().columns.tolist()
    low = {c.lower(): c for c in cols}

    doc_candidates = ['doc_id', 'docid', 'id', 'pid']
    txt_candidates = ['passage_text', 'text', 'contents', 'body', 'passage']

    doc_col = next((low[c] for c in doc_candidates if c in low), None)
    txt_col = next((low[c] for c in txt_candidates if c in low), None)

    if doc_col is None or txt_col is None:
        raise RuntimeError(
            f'Colunas nao identificadas em {parquet_glob}. Colunas disponiveis: {cols}'
        )
    return doc_col, txt_col

def fetch_passage_texts(conn, parquet_glob: str, ids_table: str = 'needed_ids') -> pd.DataFrame:
    doc_col, txt_col = detect_columns(conn, parquet_glob)
    print(f'Usando doc_col={doc_col}, text_col={txt_col} em {parquet_glob}')

    sql = f"""
        SELECT
            CAST(p.\"{doc_col}\" AS VARCHAR) AS doc_id,
            CAST(p.\"{txt_col}\" AS VARCHAR) AS passage_text
        FROM read_parquet('{parquet_glob}') p
        INNER JOIN {ids_table} n
            ON CAST(p.\"{doc_col}\" AS VARCHAR) = n.doc_id
    """
    return conn.execute(sql).df()

last_error = None
df_doc_texts = None
for parquet_path in REMOTE_PARQUET_CANDIDATES:
    try:
        print(f'Tentando fonte: {parquet_path}')
        df_doc_texts = fetch_passage_texts(con, parquet_path)
        if len(df_doc_texts) == 0:
            raise RuntimeError(f'Nenhum doc encontrado em {parquet_path}')
        print(f'Fonte selecionada: {parquet_path}')
        break
    except Exception as e:
        last_error = e
        print(f'Falhou em {parquet_path}: {e}')

if df_doc_texts is None:
    auth_hint = (
        '\nDica de autenticacao HF:\n'
        '1) Aceite os termos do dataset no Hugging Face (se gated).\n'
        '2) Gere um token Read em https://huggingface.co/settings/tokens\n'
        '3) Defina no shell e reabra o VS Code para propagar no kernel:\n'
        '   export HF_TOKEN=seu_token\n'
        '4) Reexecute da celula 3 em diante.\n'
        '5) Alternativa: defina MSMARCO_V2_PASSAGES_PARQUET com uma URL https/s3 publica.'
    )
    raise RuntimeError(
        'Nao foi possivel ler Parquet remoto. Defina MSMARCO_V2_PASSAGES_PARQUET com um glob valido.'
        + auth_hint
        + f'\nUltimo erro: {last_error}'
    )

df_doc_texts = df_doc_texts.drop_duplicates(subset=['doc_id'], keep='first')
print('Total de textos recuperados:', len(df_doc_texts))
df_doc_texts.head(3)

Total de doc_ids unicos para buscar texto: 22899
Extensao hf indisponivel, seguindo com httpfs.
SET hf_token nao suportado nesta versao de DuckDB.
Secret HUGGINGFACE configurado no DuckDB.
Tentando fonte: hf://datasets/irds/msmarco-passage-v2/docs/*.parquet
Falhou em hf://datasets/irds/msmarco-passage-v2/docs/*.parquet: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/irds/msmarco-passage-v2/tree/main/docs' (HTTP 404)

LINE 1: SELECT * FROM read_parquet('hf://datasets/irds/msmarco-passage-v2/docs...
                      ^
Tentando fonte: hf://datasets/irds/msmarco-passage-v2/**/*.parquet
Falhou em hf://datasets/irds/msmarco-passage-v2/**/*.parquet: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/irds/msmarco-passage-v2/tree/main' (HTTP 404)

LINE 1: SELECT * FROM read_parquet('hf://datasets/irds/msmarco-passage-v2/**/...
                      ^


RuntimeError: Nao foi possivel ler Parquet remoto. Defina MSMARCO_V2_PASSAGES_PARQUET com um glob valido.
Dica de autenticacao HF:
1) Aceite os termos do dataset no Hugging Face (se gated).
2) Gere um token Read em https://huggingface.co/settings/tokens
3) Defina no shell e reabra o VS Code para propagar no kernel:
   export HF_TOKEN=seu_token
4) Reexecute da celula 3 em diante.
5) Alternativa: defina MSMARCO_V2_PASSAGES_PARQUET com uma URL https/s3 publica.
Ultimo erro: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/irds/msmarco-passage-v2/tree/main' (HTTP 404)

LINE 1: SELECT * FROM read_parquet('hf://datasets/irds/msmarco-passage-v2/**/...
                      ^

In [ ]:
def finalize(df: pd.DataFrame, texts_df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(texts_df, on='doc_id', how='left')
    out = out[['query_id', 'doc_id', 'relevance_label', 'query_text', 'passage_text']].copy()

    # Enforce tipos e limpeza basica
    out['query_id'] = out['query_id'].astype(str)
    out['doc_id'] = out['doc_id'].astype(str)
    out['relevance_label'] = out['relevance_label'].astype(int)
    out['query_text'] = out['query_text'].astype(str)
    out['passage_text'] = out['passage_text'].astype('string').str.strip()

    # Remove vazios e nulos de passage_text para cumprir requisito final
    out = out[out['passage_text'].notna()]
    out = out[out['passage_text'].str.len() > 0]

    # Dedupe opcional por query-doc
    out = out.drop_duplicates(subset=['query_id', 'doc_id'], keep='first')
    return out.reset_index(drop=True)

train_final = finalize(df_train_candidates, df_doc_texts)
trec_final = finalize(df_dl23_candidates, df_doc_texts)

train_final.to_csv(TRAIN_OUT, index=False)
trec_final.to_csv(TREC_OUT, index=False)

print(f'Salvo: {TRAIN_OUT} ({len(train_final)} linhas)')
print(f'Salvo: {TREC_OUT} ({len(trec_final)} linhas)')

In [ ]:
# Validacao final: contagem e nulos em passage_text
train_check = pd.read_csv(TRAIN_OUT)
trec_check = pd.read_csv(TREC_OUT)

required_cols = ['query_id', 'doc_id', 'relevance_label', 'query_text', 'passage_text']
assert list(train_check.columns) == required_cols, f'Colunas treino invalidas: {train_check.columns}'
assert list(trec_check.columns) == required_cols, f'Colunas trec invalidas: {trec_check.columns}'

train_nulls = int(train_check['passage_text'].isna().sum())
trec_nulls = int(trec_check['passage_text'].isna().sum())

print('=== Validacao Train ===')
print('Registros:', len(train_check))
print('Nulos em passage_text:', train_nulls)

print('=== Validacao TREC DL 2023 ===')
print('Registros:', len(trec_check))
print('Nulos em passage_text:', trec_nulls)

assert train_nulls == 0, 'Treino contem nulos em passage_text'
assert trec_nulls == 0, 'TREC contem nulos em passage_text'
print('OK: ambos os arquivos estao sem nulos em passage_text.')